# MNIST Shared LoRA Pilot

Goal: test whether a single requirement-conditioned MNIST LoRA can replace one LoRA per requirement. This notebook prepares a small distillation-style training set from the existing RBT4DNN MNIST outputs, then evaluates generated images after you run training/generation in Colab.

In [1]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/casparbreloh/rbt4dnn-seminar.git"
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "data/requirements.csv").exists()), None)
if ROOT is None:
    ROOT = Path("/content/rbt4dnn-seminar")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)

EXPERIMENT = ROOT / "experiments" / "mnist-shared-lora-pilot"
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(EXPERIMENT))
print(ROOT)

/Users/casparbreloh/Developer/university/seminar/rbt4dnn-seminar


In [2]:
from shared_lora_pilot import prepare_training_data, write_template

train_dir = prepare_training_data(ROOT, max_per_requirement=100)
template = write_template(ROOT)
print("training data:", train_dir)
print("result template:", template)
print("images:", len(list((train_dir / "images").glob("*.png"))))

training data: /Users/casparbreloh/Developer/university/seminar/rbt4dnn-seminar/outputs/mnist-shared-lora-pilot/train
result template: /Users/casparbreloh/Developer/university/seminar/rbt4dnn-seminar/experiments/mnist-shared-lora-pilot/results-template.csv
images: 600


## Colab training step

Use the generated folder `outputs/mnist-shared-lora-pilot/train/` as the captioned image dataset. Train one text-conditioned LoRA with all M1-M6 prompts mixed together. After generation, save images as:

```text
outputs/mnist-shared-lora-pilot/generated/M1/*.png
outputs/mnist-shared-lora-pilot/generated/M2/*.png
...
outputs/mnist-shared-lora-pilot/generated/M6/*.png
```

Generate the same count per requirement, ideally 100 or 1000. Then rerun the evaluation cell below.

In [3]:
from pathlib import Path

print(
    "Training is intentionally left to the Colab LoRA tool you choose, because FLUX access and GPU memory vary."
)
print("Recommended output folder:", ROOT / "outputs" / "mnist-shared-lora-pilot" / "generated")
print("The next cell evaluates that folder once generated images exist.")

Training is intentionally left to the Colab LoRA tool you choose, because FLUX access and GPU memory vary.
Recommended output folder: /Users/casparbreloh/Developer/university/seminar/rbt4dnn-seminar/outputs/mnist-shared-lora-pilot/generated
The next cell evaluates that folder once generated images exist.


In [4]:
from shared_lora_pilot import generated_folders, write_results

folders = generated_folders(ROOT)
missing = [req for req, folder in folders.items() if not list(folder.glob("*.png"))]
if missing:
    print("No generated images yet for:", ", ".join(missing))
else:
    out, summary = write_results(ROOT)
    print(out)
    print(summary)

No generated images yet for: M1, M2, M3, M4, M5, M6


This pilot is the retraining extension. It only becomes a result once generated images are written back and `results.csv` is produced.